In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from lightgbm import LGBMRegressor
from sklearn.metrics import *
from sklearn.model_selection import *

import warnings
warnings.filterwarnings('ignore')

c:\Users\Dylan\Documents\School\ML Project\menv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
all_data_df = pd.read_csv("Training Data.csv")
ss = pd.read_csv("SampleSubmission.csv")

In [3]:
# Split 'Source' into 'consumer_device_X' and 'data_user_Y'
all_data_df[['consumer_device', 'data_user']] = all_data_df['Source'].str.extract(r'(consumer_device_\d+)_data_user_(\d+)')

In [4]:
# for all_data["Source"] aggregate by sum on day

# Convert 'Datetime' column to datetime objects if it's not already
all_data_df['date_time'] = pd.to_datetime(all_data_df['date_time'])

# Extract the date part
all_data_df['Date'] = all_data_df['date_time'].dt.date

# Group by 'Source' and 'Date', then sum the 'Load' for each group
aggregated_data = all_data_df.groupby(['Source', 'Date'])['kwh'].sum().reset_index()

In [5]:
# Find the minimum and maximum date_time values
min_date = aggregated_data['Date'].min()
max_date = aggregated_data['Date'].max()

print(f"Minimum date_time: {min_date}")
print(f"Maximum date_time: {max_date}")

Minimum date_time: 2023-06-03
Maximum date_time: 2024-09-23


In [6]:
# Fill missing date values with 0 kwh

# Create a date range
date_rng = pd.date_range(start='2024-07-01', end=max_date, freq='D')

# Create an empty DataFrame to store the complete data
complete_data = pd.DataFrame()

# Iterate through each unique 'Source'
for source in aggregated_data['Source'].unique():
    # Extract data for the current 'Source'
    source_data = aggregated_data[aggregated_data['Source'] == source].copy()

    # Convert the source data Date to match the type of date_rng
    source_data['Date'] = pd.to_datetime(source_data['Date'])

    # Create a complete date range for the current 'Source'
    source_date_rng = pd.DataFrame({'Date': date_rng})
    source_date_rng['Source'] = source

    # Merge with the existing data, filling missing 'kwh' values with 0
    source_data = pd.merge(source_date_rng, source_data, on=['Date', 'Source'], how='left')
    # source_data['kwh'] = source_data['kwh'].fillna(0)
    # source_data = source_data.dropna()
    # Append to the complete data
    complete_data = pd.concat([complete_data, source_data], ignore_index=True)

In [7]:
# Drop first 6 months due to missing values. Can't use log scaling to deal with right skewed data since 0s are present, not low values.
complete_data.dropna(inplace=True)

In [8]:
climate_df = pd.read_excel('Kalam Climate Data.xlsx')
# Convert to datetime
complete_data["Date"] = pd.to_datetime(complete_data["Date"])
climate_df["Date Time"] = pd.to_datetime(climate_df["Date Time"])

# Aggregate climate data to daily level
climate_daily = climate_df.groupby(climate_df["Date Time"].dt.date).agg({
    "Temperature (°C)": "mean",
    "Dewpoint Temperature (°C)": "mean",
    "U Wind Component (m/s)": "mean",
    "V Wind Component (m/s)": "mean",
    "Total Precipitation (mm)": "sum",
    "Snowfall (mm)": "sum",
    "Snow Cover (%)": "mean",
}).reset_index()

# Convert 'Date' column in climate_daily to datetime format
climate_daily.rename(columns={"Date Time": "Date"}, inplace=True)
climate_daily["Date"] = pd.to_datetime(climate_daily["Date"])  # Ensure datetime64[ns]

# Merge with complete_data
complete_data = complete_data.merge(climate_daily, on="Date", how="left")

In [9]:
# Extract consumer device and data user using regex
complete_data["consumer_device"] = complete_data["Source"].str.extract(r'consumer_device_(\d+)_data_user_\d+').astype(int)
complete_data["data_user"] = complete_data["Source"].str.extract(r'consumer_device_\d+_data_user_(\d+)').astype(int)

In [10]:
# Convert 'Date' to datetime format if not already
complete_data["Date"] = pd.to_datetime(complete_data["Date"])

# Extract date features
complete_data["day_of_week"] = complete_data["Date"].dt.dayofweek
complete_data["is_weekend"] = complete_data["day_of_week"].isin([5, 6]).astype(int)
complete_data["day_of_year"] = complete_data["Date"].dt.dayofyear
# Weekly/Monthly Aggregation
complete_data["Week"] = complete_data["Date"].dt.isocalendar().week
complete_data["Month"] = complete_data["Date"].dt.month

In [11]:
complete_data["temp_dew_diff"] = complete_data["Temperature (°C)"] - complete_data["Dewpoint Temperature (°C)"] # Gauges humidity
complete_data["wind_speed"] = (complete_data["U Wind Component (m/s)"]**2 + complete_data["V Wind Component (m/s)"]**2)**0.5 # Wind speed

complete_data["precip_decay"] = (
    (1 / np.log(2)) * complete_data["Total Precipitation (mm)"].shift(1) +
    (1 / np.log(3)) * complete_data["Total Precipitation (mm)"].shift(2) +
    (1 / np.log(4)) * complete_data["Total Precipitation (mm)"].shift(3)) # Adds rate decay for precip
first_valid = complete_data["precip_decay"].dropna().iloc[0]
complete_data["precip_decay"].fillna(value=first_valid, inplace=True)

complete_data["snow_decay"] = (
    (1 / np.log(2)) * complete_data["Snowfall (mm)"].shift(1) +
    (1 / np.log(3)) * complete_data["Snowfall (mm)"].shift(2) +
    (1 / np.log(4)) * complete_data["Snowfall (mm)"].shift(3))
first_valid = complete_data["snow_decay"].dropna().iloc[0]
complete_data["snow_decay"].fillna(value=first_valid, inplace=True)

In [12]:
complete_data = complete_data.sort_values(by=["Source", "Date"]).reset_index(drop=True)

In [13]:
# Unique sources (users/devices)
complete_data["Source"] = complete_data["Source"].astype("category")
print("Unique Devices/Users:", complete_data["Source"].nunique())

Unique Devices/Users: 494


In [14]:
X = complete_data.drop(columns=['Date', 'kwh', 'Month', 'Week', 'Snow Cover (%)', 'is_weekend','consumer_device','data_user'])
y = complete_data.kwh

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42,
                                                    shuffle=True,test_size=0.2,
                                                    stratify=X['Source']
                                                   )

In [16]:
model = LGBMRegressor(
    n_estimators=1000,
    objective='rmse',
    random_state=42,
    early_stopping_rounds=100,
    verbose=100
)

model.fit(
X_train, y_train,
    eval_set=[(X_test, y_test)],
       categorical_feature=['Source']
)

print()
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.5f}")
print(f"MSE:  {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"R²:   {r2:.5f}")

[LightGBM] [Warning] early_stopping_round is set=100, early_stopping_rounds=100 will be ignored. Current value: early_stopping_round=100
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.844515
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.141751
[LightGBM] [Debug] init for col-wise cost 0.000562 seconds, init for row-wise cost 0.000538 seconds
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000870 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Debug] Using Dense Multi-Val Bin
[LightGBM] [Info] Total 

In [ ]:
def objective(trial):
    # Suggest hyperparameters
    params = {
        'n_estimators': 1000,
        'objective': 'rmse',
        'random_state': 42,
        'early_stopping_rounds': 100,
        'verbose': 100,
        'num_leaves': trial.suggest_int('num_leaves', 20, 40),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-5, 1e-1),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'lambda_l1': trial.suggest_loguniform('lambda_l1', 1e-5, 10.0),
        'lambda_l2': trial.suggest_loguniform('lambda_l2', 1e-5, 10.0),
    }

    # Initialize and train the model with the suggested parameters
    model = LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        categorical_feature=['Source']
    )

    # Predict and evaluate
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    return rmse

# Create an Optuna study to optimize the objective function
study = optuna.create_study(direction='minimize')  # Minimize RMSE
study.optimize(objective, n_trials=500)  # Number of trials to search for the best hyperparameters

# Print the best trial and its parameters
print('Best trial:')
best_trial = study.best_trial
print(f'RMSE: {best_trial.value}')
print('Best hyperparameters:')
for key, value in best_trial.params.items():
    print(f'{key}: {value}')

In [18]:
# Use the best hyperparameters to train the final model
best_params = best_trial.params
best_model = LGBMRegressor(**best_params)
best_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=['Source']
)

# Evaluate the final model
y_pred = best_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print()
print(f"MAE:  {mae:.5f}")
print(f"MSE:  {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"R²:   {r2:.5f}")

[LightGBM] [Warning] lambda_l1 is set=0.032390313329348844, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.032390313329348844
[LightGBM] [Warning] lambda_l2 is set=2.727510286101453e-05, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.727510286101453e-05
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] lambda_l1 is set=0.032390313329348844, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.032390313329348844
[LightGBM] [Warning] lambda_l2 is set=2.727510286101453e-05, reg_lambda=0.0 will be ignored. Current value: lambda_l2=2.727510286101453e-05
[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.844515
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures:

INFERENCE


In [19]:
ss = pd.read_csv('SampleSubmission.csv')

In [20]:
# Extract date part from ID column
ss["Date"] = pd.to_datetime(ss["ID"].str.extract(r'(\d{4}-\d{2}-\d{2})')[0])

In [21]:
# Convert to datetime
ss["Date"] = pd.to_datetime(ss["Date"])
climate_df["Date Time"] = pd.to_datetime(climate_df["Date Time"])

# Aggregate climate data to daily level
climate_daily = climate_df.groupby(climate_df["Date Time"].dt.date).agg({
    "Temperature (°C)": "mean",
    "Dewpoint Temperature (°C)": "mean",
    "U Wind Component (m/s)": "mean",
    "V Wind Component (m/s)": "mean",
    "Total Precipitation (mm)": "sum",
    "Snowfall (mm)": "sum",
    "Snow Cover (%)": "mean",
}).reset_index()

# Convert 'Date' column in climate_daily to datetime format
climate_daily.rename(columns={"Date Time": "Date"}, inplace=True)
climate_daily["Date"] = pd.to_datetime(climate_daily["Date"])  # Ensure datetime64[ns]

# Merge with complete_data
forecast = ss.merge(climate_daily, on="Date", how="left")

In [22]:
# Split ID into Date and Source
forecast[["Date", "device", "data", "user"]] = forecast["ID"].str.split("_", expand=True, n=3)

# Convert Date to datetime format
forecast["Date"] = pd.to_datetime(forecast["Date"])

# Reconstruct Source (combine device, data, and user)
forecast["Source"] = forecast["device"] + "_" + forecast["data"] + "_" + forecast["user"]

In [23]:
# Extract consumer device and data user using regex
forecast["consumer_device"] = forecast["ID"].str.extract(r'consumer_device_(\d+)_data_user_\d+').astype(int)
forecast["data_user"] = forecast["ID"].str.extract(r'consumer_device_\d+_data_user_(\d+)').astype(int)

In [24]:
# Drop unnecessary columns
forecast.drop(columns=["device", "data", "user", "ID"], inplace=True)

In [25]:
forecast["Source"] = forecast["Source"].astype("category")

In [33]:
forecast["temp_dew_diff"] = forecast["Temperature (°C)"] - forecast["Dewpoint Temperature (°C)"] # Gauges humidity
forecast["wind_speed"] = (forecast["U Wind Component (m/s)"]**2 + forecast["V Wind Component (m/s)"]**2)**0.5 # Wind speed

forecast["precip_decay"] = (
    (1 / np.log(2)) * forecast["Total Precipitation (mm)"].shift(1) +
    (1 / np.log(3)) * forecast["Total Precipitation (mm)"].shift(2) +
    (1 / np.log(4)) * forecast["Total Precipitation (mm)"].shift(3)) # Adds rate decay for precip
first_valid = forecast["precip_decay"].dropna().iloc[0]
forecast["precip_decay"].fillna(value=first_valid, inplace=True)

forecast["snow_decay"] = (
    (1 / np.log(2)) * forecast["Snowfall (mm)"].shift(1) +
    (1 / np.log(3)) * forecast["Snowfall (mm)"].shift(2) +
    (1 / np.log(4)) * forecast["Snowfall (mm)"].shift(3))
first_valid = forecast["snow_decay"].dropna().iloc[0]
forecast["snow_decay"].fillna(value=first_valid, inplace=True)

In [34]:
# Conforecastvert 'Date' to datetime format if not already
forecast["Date"] = pd.to_datetime(forecast["Date"])

# Extract date features
forecast["year"] = forecast["Date"].dt.year
forecast["month"] = forecast["Date"].dt.month
forecast["day"] = forecast["Date"].dt.day
forecast["day_of_week"] = forecast["Date"].dt.dayofweek  # Monday=0, Sunday=6
forecast["week_of_year"] = forecast["Date"].dt.isocalendar().week
forecast["quarter"] = forecast["Date"].dt.quarter
forecast["is_weekend"] = (forecast["day_of_week"] >= 5).astype(int)  # 1 if Sat/Sun, else 0

In [35]:
forecast["day_of_week"] = forecast["Date"].dt.dayofweek
forecast["is_weekend"] = forecast["day_of_week"].isin([5, 6]).astype(int)
forecast["day_of_year"] = forecast["Date"].dt.dayofyear

In [36]:
# Weekly/Monthly Aggregation
forecast["Week"] = forecast["Date"].dt.isocalendar().week
forecast["Month"] = forecast["Date"].dt.month

In [ ]:
X_test_final = forecast.drop(columns=['Date', 'kwh', 'month', 'Week', 'Snow Cover (%)', 'is_weekend', 'quarter',  'week_of_year', 'year'])

In [45]:
X_test_final

,Temperature (°C),Dewpoint Temperature (°C),U Wind Component (m/s),V Wind Component (m/s),Total Precipitation (mm),Snowfall (mm),Source,consumer_device,data_user,temp_dew_diff,wind_speed,precip_snow_ratio,year,day,day_of_week,day_of_year,Month,precip_decay,snow_decay
0,13.899341,2.104299,0.005811,-0.264604,0.000032,0.000000,consumer_device_12_data_user_1,12,1,11.795041,0.264667,31.505000,2024,24,1,268,9,0.180461,0.000000e+00
1,12.475849,5.623678,0.224280,0.183461,0.003789,0.000000,consumer_device_12_data_user_1,12,1,6.852172,0.289758,3789.191000,2024,25,2,269,9,0.180461,0.000000e+00
2,9.702699,7.375160,0.202651,0.044908,0.122680,0.000000,consumer_device_12_data_user_1,12,1,2.327538,0.207568,122679.821000,2024,26,3,270,9,0.180461,0.000000e+00
3,6.806661,5.905107,0.042285,0.168467,0.443926,0.019848,consumer_device_12_data_user_1,12,1,0.901553,0.173692,22.365588,2024,27,4,271,9,0.180461,0.000000e+00
4,6.399286,2.932036,-0.042425,0.149797,0.059494,0.001262,consumer_device_12_data_user_1,12,1,3.467250,0.155689,47.097240,2024,28,5,272,9,0.754850,2.863403e-02
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6009,3.693599,-4.460602,0.046748,-0.296137,0.003672,0.000000,consumer_device_8_data_user_9,8,9,8.154201,0.299804,3671.920000,2024,20,6,294,10,0.032725,1.402936e-02
6010,3.712101,-3.217974,-0.118125,-0.373651,0.008551,0.000000,consumer_device_8_data_user_9,8,9,6.930075,0.391879,8551.200000,2024,21,0,295,10,0.009118,8.620140e-04
6011,2.515382,-2.358298,0.047106,-0.214380,0.010362,0.000161,consumer_device_8_data_user_9,8,9,4.873680,0.219495,63.956484,2024,22,1,296,10,0.015860,8.295496e-07
6012,0.315618,-0.711822,0.001728,-0.227600,0.056409,0.027086,consumer_device_8_data_user_9,8,9,1.027440,0.227606,2.082501,2024,23,2,297,10,0.025381,2.322883e-04


Now let's predict on the test set since we have all the features necessary

In [32]:
predictions = model.predict(X_test_final[X.columns])

KeyError: "['precip_decay', 'snow_decay'] not in index"

In [ ]:
forecast = pd.read_csv('SampleSubmission.csv')

In [ ]:
forecast['kwh'] = predictions.clip(min=0)

In [ ]:
forecast.to_csv("sample_submission_new.csv", index = False)